# WP4v3 — Notebook 2 : Entraînement

**Normalisation cohérente** :
- Stats calculées sur `cls_mae` du **train uniquement** (notebook 1)
- Appliquées à train ET val avant tout forward pass
- La loss et les métriques sont toujours calculées sur des données normalisées

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')


In [ ]:
data_train = torch.load('wp4v3_pairs_train.pt')
data_val   = torch.load('wp4v3_pairs_val.pt')

# Charger les stats CLS calculees sur le train (notebook 1)
cls_norm = torch.load('wp4v3_cls_norm_stats.pt')
cls_mean = cls_norm['mean']  # (1024,)
cls_std  = cls_norm['std']   # (1024,)

# Normaliser train ET val avec les MEMES stats train
z_train = (data_train['cls_mae'] - cls_mean) / cls_std  # (N_train, 1024)
z_val   = (data_val['cls_mae']   - cls_mean) / cls_std  # (N_val,   1024)

# Verification : mean~0, std~1 sur train ; approximativement sur val
print(f'z_train — mean: {z_train.mean():.4f}, std: {z_train.std():.4f}')
print(f'z_val   — mean: {z_val.mean():.4f},   std: {z_val.std():.4f}')
print(f'cls_clip norm moyenne (train): {data_train["cls_clip"].norm(dim=-1).mean():.4f}')  # ~1.0

BATCH_SIZE   = 256
train_loader = DataLoader(TensorDataset(z_train, data_train['cls_clip']),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(TensorDataset(z_val,   data_val['cls_clip']),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'Train : {len(z_train)} | Val : {len(z_val)}')


In [ ]:
class ProjectionMLP(nn.Module):
    """
    f_theta : cls_mae normalise (1024) -> espace CLS CLIP (1024)
    Architecture plus profonde : 3 couches lineaires, hidden_dim=2048
    """
    def __init__(self, dim=1024, hidden_dim=2048, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(dim),
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, dim),
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

f_theta = ProjectionMLP().to(DEVICE)
print(f'Parametres : {sum(p.numel() for p in f_theta.parameters()):,}')
print(f'Architecture : 1024 -> 2048 -> 2048 -> 1024')


In [ ]:
N_EPOCHS, LR, WEIGHT_DECAY, PATIENCE = 200, 1e-3, 1e-3, 20
optimizer = AdamW(f_theta.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
history   = {'train_loss': [], 'val_loss': [], 'val_cos': []}
best_cos, best_epoch, patience_counter = 0., 0, 0

def cosine_loss(a, b): return (1 - F.cosine_similarity(a, b)).mean()

def eval_epoch(loader):
    """Evaluation sur donnees deja normalisees."""
    f_theta.eval()
    tl, tc, n = 0., 0., 0
    with torch.no_grad():
        for z, c in loader:
            z, c = z.to(DEVICE), c.to(DEVICE)
            # z est deja normalise — on le passe directement a f_theta
            p = f_theta(z)
            tl += cosine_loss(p, c).item() * z.shape[0]
            tc += F.cosine_similarity(p, c).mean().item() * z.shape[0]
            n  += z.shape[0]
    return tl/n, tc/n

for epoch in range(N_EPOCHS):
    f_theta.train()
    tl, n = 0., 0
    for z, c in train_loader:
        z, c = z.to(DEVICE), c.to(DEVICE)
        optimizer.zero_grad()
        loss = cosine_loss(f_theta(z), c)
        loss.backward()
        optimizer.step()
        tl += loss.item() * z.shape[0]; n += z.shape[0]
    scheduler.step()

    train_loss = tl/n
    val_loss, val_cos = eval_epoch(val_loader)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_cos'].append(val_cos)

    if val_cos > best_cos:
        best_cos, best_epoch, patience_counter = val_cos, epoch+1, 0
        torch.save(f_theta.state_dict(), 'wp4v3_ftheta_best.pt')
    else:
        patience_counter += 1

    if (epoch+1) % 20 == 0:
        print(f'Epoch {epoch+1:3d} | train={train_loss:.4f} | val={val_loss:.4f} | '
              f'cos={val_cos:.4f} | patience={patience_counter}/{PATIENCE}')
    if patience_counter >= PATIENCE:
        print(f'Early stopping a l epoch {epoch+1}'); break

print(f'Meilleur modele : epoch {best_epoch}, val_cos={best_cos:.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'], label='val')
axes[0].axvline(best_epoch-1, color='red', linestyle='--', alpha=0.5, label=f'best={best_epoch}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss cosinus')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history['val_cos'], color='darkorange')
axes[1].axvline(best_epoch-1, color='red', linestyle='--', alpha=0.5, label=f'best={best_cos:.4f}')
axes[1].axhline(1.0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Cos. sim.')
axes[1].set_title('Alignement CLS MAE -> CLS CLIP'); axes[1].legend()
plt.tight_layout()
plt.savefig('wp4v3_training_curves.png', dpi=150)
plt.show()
